In [ ]:
# MCC - Temas Selectos III 
# Instructor: Dr. J.A. Garcia-Rodriguez
# PART-01

# Imports the main libraries and defines general settings for reproducibility,
# data display, warnings, and time-series data handling.
import warnings
warnings.filterwarnings("ignore")

import os
os.environ["NIXTLA_ID_AS_COL"] = "true"

import numpy as np
np.set_printoptions(suppress=True)
np.random.seed(1)

import random
random.seed(1)

import pandas as pd
pd.set_option("max_colwidth", 100)
pd.set_option("display.precision", 3)


In [ ]:
# Install fpppy, a package with tools for time-series analysis and forecasting.
%pip install fpppy

In [ ]:
# Imports a utility function for visualizing time-series data.
from utilsforecast.plotting import plot_series as plot_series_utils

In [ ]:
#Install Seaborn, a Python library for statistical data visualization.
%pip install seaborn

In [ ]:
# Configures Matplotlib and Seaborn for time-series visualization and
# defines custom color palettes and colormaps for consistent plotting.

import seaborn as sns
sns.set_style("whitegrid")

import matplotlib.pyplot as plt
plt.style.use("ggplot")
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "figure.constrained_layout.use": True,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "legend.title_fontsize": 10,
    "grid.alpha": 1.0,
})

import matplotlib as mpl

from cycler import cycler
mpl.rcParams['axes.prop_cycle'] = cycler(color=["#000000", "#000000"])

from fpppy.utils import plot_series
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#2f2fff"], name="black_and_blue"),
    force=True,
)

mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#D55E00"], name="black_and_orange"),
    force=True,
)

mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#000000"], name="black"),
    force=True,
)

mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#0072B2", "#D55E00"],
        name='black_and_2color',
    ),
    force=True
)

mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#D55E00", "#0072B2", "#009E73"],
        name='black_and_3color',
    ),
    force=True
)

mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#D55E00", "#0072B2", "#009E73", "#CC79A7"],
        name='black_and_4color',
    ),
    force=True
)

mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#D55E00", "#0072B2", "#009E73", "#CC79A7"],
        name='r_colors',
    ),
    force=True
)

In [ ]:
import statsmodels.api as sm
from scipy.stats import pearsonr
from statsmodels.graphics.tsaplots import plot_acf

In [ ]:
df = pd.DataFrame({
    "Year": list(range(2015, 2020)),
    "Observation": [123, 39, 78, 52, 110],
})
df

In [ ]:
print(df.index)
print(df.columns)

In [ ]:
df.dtypes

In [ ]:
print(type(df["Year"]))
print(df["Year"])

In [ ]:
year_df = df.set_index("Year")
year_df

In [ ]:
# Time series observations may be associated with instants in time or spans of time. 
# Pandas supports these concepts through pd.Timestamp and pd.Period classes.

print(repr(pd.Timestamp("2020-01")))
print(repr(pd.Period("2020-01")))

In [ ]:
# Pandas provides many utilities for converting between text strings 
# and dedicated timestamp or period representations.

print(repr(pd.Timestamp("2020")))
print(repr(pd.Timestamp("2020-01-01 12:34")))
print(repr(pd.Period("2020-01-01")))
print(repr(pd.Period("2020-01-01", freq="M")))

In [ ]:
# Timestamp sequences can be conveniently constructed using
# pd.to_datetime() or pd.date_range()

ts_few = pd.to_datetime(["2020-01-01", "2020-01-02", "2020-01-03"])
ts_range = pd.date_range("2020-01-01", "2020-01-02", freq="12h")
print(ts_few)
print(ts_range)

In [ ]:
# The .to_period() and .to_timestamp() methods allow conversions
# between timestamp and period. 
# The .to_period() method will infer the period duration unless the freq= argument is specified.

print(ts_range.to_period())
print(ts_range.to_period(freq="D"))
print(ts_range.to_period(freq="W"))
print(ts_range.to_period().to_timestamp())

In [ ]:
#Both timestamps and period start times can be converted to strings with custom formatting
print(ts_few.strftime("%m/%d/%Y"))
print(ts_range.to_period().strftime("%Y %b ~ %H:%M"))

In [ ]:
# Pandas automatically converts between Index and Series types 
# as needed when these sequences are used as the data or index in a DataFrame.

df = pd.DataFrame({"ts": ts_few})
df = df.assign(
    period=df["ts"].dt.to_period(),
    yr=df["ts"].dt.year,
    str=df["ts"].dt.strftime("%A, %B %#d")).set_index("ts")
df

In [ ]:
# PART 02 - DATASETS
# Suppose you are interested in a dataset containing the fastest running times for 
# women’s and men’s races at the Olympics, from 100m to 10000m
# This DataFrame contains 312 rows and 4 columns. Data is recorded every four years.

olympic_running = pd.read_csv("01_Running_data.csv")
olympic_running.head(7)

In [ ]:
# A missing value in 1916 becuse the Olympics were not held during WWI.
# The 14 time series are uniquely identified by the Length and Sex key variables.

print(olympic_running["Sex"].unique())
print(olympic_running["Length"].unique())

In [ ]:
print(olympic_running[["Sex", "Length"]].drop_duplicates())

In [ ]:
# Pharmaceutical Benefits Scheme (PBS) dataset is a 
# time series dataset that contains  information about 
# the number of prescriptions dispensed under PBS in Australia. 
# parse_date forces datetime strings into datetime objects.

pbs = pd.read_csv("02_pbs_data.csv", parse_dates=["Month"])
pbs = pbs[["Month", "Concession", "Type", "ATC1", "ATC2", "Scripts", "Cost"]]
pbs


In [ ]:
# Set contains monthly data on Medicare Australia prescription
# data from July 1991 to June 2008. 
# These are classified according to various concession types, 
# and Anatomical Therapeutic Chemical (ATC) indexes.

a10 = pbs.loc[pbs["ATC2"] == "A10"]
a10

In [ ]:
a10 = (pbs.loc[pbs["ATC2"] == "A10"].drop(columns=["ATC1", "ATC2"]))
a10.head(25)

In [ ]:
# The new column, which we named TotalC using the .rename() method, 
# represents the sum of all Cost values for each month.

total_cost_df = (
    pbs.loc[pbs["ATC2"] == "A10"]
    .drop(columns=["ATC1", "ATC2"])
    .groupby("Month", as_index=False)
    .agg({"Cost": "sum"})
    .rename(columns={"Cost": "TotalC"})
)
total_cost_df


In [ ]:
# The new column, which we named TotalC using the .assign() method,
# represents the sum of all Cost values for each month converted to millions and rounded to two decimal places.

total_cost_df = (
    pbs.loc[pbs["ATC2"] == "A10"]
    .drop(columns=["ATC1", "ATC2"])
    .groupby("Month", as_index=False)
    .agg({"Cost": "sum"})
    .assign(Cost=lambda x: (x["Cost"] / 1e6).round(2))
)
total_cost_df



In [ ]:
total_cost_df.to_csv("02_total_cost.csv", index=False) 

In [ ]:
# The prison dataset contains quarterly data on the number of prisoners in Australia,
# broken down by state, gender, legal status, and indigenous status. 
# Notes: ATSI stands for Aboriginal and Torres Strait Islander. 
# Count is the number of prisoners in each category.

prison = (pd.read_csv("03_prison.csv", parse_dates=["Date"]))
prison

In [ ]:
prison = (pd.read_csv("03_prison.csv", parse_dates=["Date"])
    .rename(columns={"Date": "Quarter"})
    .sort_values(by=["State", "Gender", "Legal", "Indigenous"])
)
prison

In [ ]:
# PART 03 - PLOTS

syd_economy = (pd.read_csv("04_syd_economy.csv", parse_dates=["ds"])
    .loc[lambda x: (x["Airports"] == "MEL-SYD") & (x["Class"] == "Economy")]
    .rename(columns={"Airports": "unique_id"})
    .assign(y=lambda x: x["y"] / 1000)
)

plot_series(df=syd_economy, id_col="unique_id", time_col="ds", target_col="y",
    xlabel="Week [1-Week long]", ylabel="Passengers ('000)",
    title="Ansett airlines economy class: Melbourne-Sydney")

In [ ]:
plot_series(total_cost_df.assign(unique_id="total_cost"),
    time_col="Month", target_col="Cost",
    xlabel="Month [1M]", ylabel=" USD millions",
    title="Australian antidiabetic drug sales")